# Agent 2 — Silver and Gold Data Generator

Generates synthetic Silver and Gold tables by applying
transformation rules to the existing Raw pilot data.

Injects controlled reconciliation discrepancies into
specific batch dates for Agent 2 testing.

Run this notebook ONCE after the Raw data generation
notebooks have been run.

Reads from:
  uc1_raw_store_transactions
  uc1_raw_inventory_movements

Writes to:
  uc2_silver_store_transactions
  uc2_silver_inventory_movements
  uc2_gold_daily_store_metrics
  uc2_gold_daily_inventory_summary

In [0]:
from datetime import date, datetime, timedelta
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DateType, TimestampType,
    DoubleType, IntegerType, LongType,
    BooleanType, FloatType
)
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ── Catalog and Schema ─────────────────────────────────────────────────────

dbutils.widgets.text("catalog", "learning_dev", "Target Catalog")
dbutils.widgets.text("schema", "s819534", "Target Schema")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")

# ── Source (Raw) Tables ────────────────────────────────────────────────────

RAW_TRANSACTIONS = f"{CATALOG}.{SCHEMA}.uc1_raw_store_transactions"
RAW_INVENTORY    = f"{CATALOG}.{SCHEMA}.uc1_raw_inventory_movements"

# ── Target (Silver) Tables ─────────────────────────────────────────────────

SILVER_TRANSACTIONS = f"{CATALOG}.{SCHEMA}.uc2_silver_store_transactions"
SILVER_INVENTORY    = f"{CATALOG}.{SCHEMA}.uc2_silver_inventory_movements"

# ── Target (Gold) Tables ───────────────────────────────────────────────────

GOLD_STORE_METRICS  = f"{CATALOG}.{SCHEMA}.uc2_gold_daily_store_metrics"
GOLD_INV_SUMMARY    = f"{CATALOG}.{SCHEMA}.uc2_gold_daily_inventory_summary"

# ── Pipeline Version ───────────────────────────────────────────────────────

PIPELINE_VERSION = "agent2-poc-v1.0"

# ── Date Configuration ─────────────────────────────────────────────────────

HISTORICAL_START = date(2026, 5, 19)
HISTORICAL_END   = date(2026, 6, 17)

HISTORICAL_DATES = [
    HISTORICAL_START + timedelta(days=i)
    for i in range((HISTORICAL_END - HISTORICAL_START).days + 1)
]

BATCH_DATES = {
    "clean"         : date(2026, 6, 19),  # All checks PASS
    "raw_issue"     : date(2026, 6, 20),  # Agent 1: Volume drop + Malformed
    "silver_drop"   : date(2026, 6, 21),  # Inject: Silver unexpected drop
    "gold_mismatch" : date(2026, 6, 22),  # Inject: Gold aggregate mismatch
    "cascading"     : date(2026, 6, 23),  # Inject: Cascading raw issue
    "freshness_issue": date(2026, 6, 24), # Inject: Freshness + combined
}

print("=" * 65)
print("AGENT 2 — SILVER AND GOLD DATA GENERATOR")
print("=" * 65)
print(f"Catalog  : {CATALOG}")
print(f"Schema   : {SCHEMA}")
print()
print(f"Historical period:")
print(f"  Start : {HISTORICAL_START}")
print(f"  End   : {HISTORICAL_END}")
print(f"  Days  : {len(HISTORICAL_DATES)}")
print()
print("Controlled batch dates:")
for name, dt in BATCH_DATES.items():
    print(f"  {name:<20} {dt}")
print("=" * 65)


## Silver Transformation Functions

In [0]:
def transform_raw_to_silver_transactions(
    business_date         : date,
    extra_drop_rate       : float = 0.0,
    load_timestamp_offset : int   = 0
) -> DataFrame:
    """
    Applies Silver transformation rules to Raw transactions.

    Transformation rules:
      1. Deduplicate on transaction_id (keep first occurrence)
      2. Exclude VOID transaction_type records
      3. Derive net_amount where null using gross - discount + tax
      4. Exclude records with null store_id
      5. Standardise store_region to UPPER

    Args:
      business_date:         Date to transform
      extra_drop_rate:       Additional records to drop (0.0 to 1.0)
                             Used to inject Silver-specific anomalies
      load_timestamp_offset: Hours to add to load timestamp
                             Used to inject freshness issues
    """
    raw = spark.table(RAW_TRANSACTIONS).filter(
        F.col("transaction_date") == F.lit(str(business_date))
    )

    raw_count = raw.count()
    print(f"    Raw records for {business_date}: {raw_count}")

    if raw_count == 0:
        print(f"    No raw data found for {business_date}. Skipping.")
        return None

    # Rule 1: Deduplicate on transaction_id
    silver = raw.dropDuplicates(["transaction_id"])

    # Rule 2: Exclude VOID transactions
    silver = silver.filter(F.col("transaction_type") != "VOID")

    # Rule 3: Derive net_amount where null
    silver = silver.withColumn(
        "net_amount",
        F.when(
            F.col("net_amount").isNull(),
            F.col("gross_amount") -
            F.coalesce(F.col("discount_amount"), F.lit(0.0)) +
            F.coalesce(F.col("tax_amount"), F.lit(0.0))
        ).otherwise(F.col("net_amount"))
    ).withColumn(
        "is_derived_net_amount",
        F.col("net_amount").isNull().cast("boolean")
    )

    # Rule 4: Exclude null store_id
    silver = silver.filter(F.col("store_id").isNotNull())

    # Rule 5: Standardise store_region
    silver = silver.withColumn(
        "store_region",
        F.upper(F.col("store_region"))
    )

    # Inject additional drop if configured (for Silver anomaly testing)
    if extra_drop_rate > 0.0:
        keep_rate = 1.0 - extra_drop_rate
        silver = silver.sample(fraction=keep_rate, seed=42)
        print(
            f"    Injected additional {extra_drop_rate*100:.0f}% drop "
            f"for anomaly testing"
        )

    # Add Silver audit columns
    load_ts = datetime.now() + timedelta(hours=load_timestamp_offset)

    silver = (
        silver
        .withColumn("is_derived_net_amount",
                    F.coalesce(
                        F.col("is_derived_net_amount"),
                        F.lit(False)
                    ))
        .withColumn("quality_score",    F.lit(1.0))
        .withColumn("silver_load_timestamp",
                    F.lit(load_ts).cast(TimestampType()))
        .withColumn("pipeline_version", F.lit(PIPELINE_VERSION))
        .drop("_rescued_data")
    )

    silver_count = silver.count()
    survival     = silver_count / raw_count if raw_count > 0 else 0

    print(f"    Silver records:  {silver_count}")
    print(f"    Survival rate:   {survival:.2%}")

    return silver


def transform_raw_to_silver_inventory(
    business_date         : date,
    extra_drop_rate       : float = 0.0,
    load_timestamp_offset : int   = 0
) -> DataFrame:
    """
    Applies Silver transformation rules to Raw inventory movements.

    Transformation rules:
      1. Filter out DELETE cdc_operation records
      2. Deduplicate on event_id keeping latest cdc_sequence_number
      3. Filter out malformed records (_rescued_data IS NOT NULL)
      4. Exclude records where quantity_after < 0
      5. Exclude records where event_id IS NULL
    """
    raw = spark.table(RAW_INVENTORY).filter(
        F.col("business_date") == F.lit(str(business_date))
    )

    raw_count = raw.count()
    print(f"    Raw records for {business_date}: {raw_count}")

    if raw_count == 0:
        print(f"    No raw data found for {business_date}. Skipping.")
        return None

    # Rule 1: Filter out DELETE operations
    silver = raw.filter(F.col("cdc_operation") != "DELETE")

    # Rule 2: Deduplicate on event_id keeping latest sequence
    from pyspark.sql.window import Window
    window = (
        Window
        .partitionBy("event_id")
        .orderBy(F.col("cdc_sequence_number").desc())
    )
    silver = (
        silver
        .withColumn("row_num", F.row_number().over(window))
        .filter(F.col("row_num") == 1)
        .drop("row_num")
        .withColumn("is_deduped", F.lit(True))
    )

    # Rule 3: Filter out malformed records
    silver = silver.filter(F.col("_rescued_data").isNull())

    # Rule 4: Exclude negative quantity_after
    silver = silver.filter(
        F.col("quantity_after").isNull() |
        (F.col("quantity_after") >= 0)
    )

    # Rule 5: Exclude null event_id
    silver = silver.filter(F.col("event_id").isNotNull())

    # Inject additional drop if configured
    if extra_drop_rate > 0.0:
        keep_rate = 1.0 - extra_drop_rate
        silver = silver.sample(fraction=keep_rate, seed=42)
        print(
            f"    Injected additional {extra_drop_rate*100:.0f}% drop "
            f"for anomaly testing"
        )

    # Add Silver audit columns
    load_ts = datetime.now() + timedelta(hours=load_timestamp_offset)

    silver = (
        silver
        .withColumn("is_deduped",
                    F.coalesce(F.col("is_deduped"), F.lit(True)))
        .withColumn("quality_flag",         F.lit("CLEAN"))
        .withColumn("silver_load_timestamp",
                    F.lit(load_ts).cast(TimestampType()))
        .withColumn("pipeline_version",     F.lit(PIPELINE_VERSION))
        .drop("_rescued_data")
    )

    silver_count = silver.count()
    survival     = silver_count / raw_count if raw_count > 0 else 0

    print(f"    Silver records:  {silver_count}")
    print(f"    Survival rate:   {survival:.2%}")

    return silver


print("Silver transformation functions loaded.")

## Gold Aggregation Functions

In [0]:
def aggregate_silver_to_gold_transactions(
    business_date         : date,
    amount_multiplier     : float = 1.0,
    load_timestamp_offset : int   = 0
) -> DataFrame:
    """
    Aggregates Silver transactions to Gold store-level daily metrics.
    """
    silver = spark.table(SILVER_TRANSACTIONS).filter(
        F.col("transaction_date") == F.lit(str(business_date))
    )

    silver_count = silver.count()
    print(f"    Silver records for {business_date}: {silver_count}")

    if silver_count == 0:
        print(f"    No silver data for {business_date}. Skipping.")
        return None

    # ── Capture silver_load_timestamp BEFORE groupBy ───────────────────────
    # After groupBy this column no longer exists in the DataFrame
    # so we must extract it first as a scalar value
    silver_ts_row = silver.agg(
        F.max("silver_load_timestamp").alias("max_ts")
    ).collect()[0]
    silver_ts_str = str(silver_ts_row["max_ts"])

    load_ts_str   = str(datetime.now() + timedelta(hours=load_timestamp_offset))

    # ── Aggregate ──────────────────────────────────────────────────────────
    gold = silver.groupBy(
        "store_id",
        "store_region",
        "transaction_date"
    ).agg(
        F.count("*")
         .cast(LongType())
         .alias("total_transactions"),
        (F.sum("gross_amount") * amount_multiplier)
         .alias("total_gross_amount"),
        (F.sum(F.coalesce("discount_amount", F.lit(0.0))) * amount_multiplier)
         .alias("total_discount_amount"),
        (F.sum("net_amount") * amount_multiplier)
         .alias("total_net_amount"),
        (F.sum(F.coalesce("tax_amount", F.lit(0.0))) * amount_multiplier)
         .alias("total_tax_amount"),
        F.avg("net_amount")
         .alias("avg_basket_value"),
        F.sum(F.coalesce("item_count", F.lit(0)))
         .cast(LongType())
         .alias("total_items_sold"),
        F.countDistinct("customer_id")
         .cast(LongType())
         .alias("unique_customers"),
        F.sum(
            F.when(F.col("transaction_type") == "RETURN", 1)
             .otherwise(0)
        ).cast(LongType())
         .alias("return_count")
    )

    # ── Add derived columns ────────────────────────────────────────────────
    gold = gold \
        .withColumnRenamed("transaction_date", "business_date") \
        .withColumn(
            "return_rate",
            F.col("return_count").cast(DoubleType()) /
            F.col("total_transactions").cast(DoubleType())
        ).withColumn(
            "silver_load_timestamp",
            F.lit(silver_ts_str).cast(TimestampType())
        ).withColumn(
            "gold_load_timestamp",
            F.lit(load_ts_str).cast(TimestampType())
        ).withColumn(
            "pipeline_version",
            F.lit(PIPELINE_VERSION)
        )

    if amount_multiplier != 1.0:
        print(
            f"    Injected amount multiplier {amount_multiplier:.2f} "
            f"for Gold aggregate mismatch testing"
        )

    gold_count = gold.count()
    print(f"    Gold rows: {gold_count} (one per store per date)")

    return gold


def aggregate_silver_to_gold_inventory(
    business_date         : date,
    load_timestamp_offset : int = 0
) -> DataFrame:
    """
    Aggregates Silver inventory to Gold warehouse-level daily summary.
    """
    silver = spark.table(SILVER_INVENTORY).filter(
        F.col("business_date") == F.lit(str(business_date))
    )

    silver_count = silver.count()
    print(f"    Silver inventory for {business_date}: {silver_count}")

    if silver_count == 0:
        print(f"    No silver inventory for {business_date}. Skipping.")
        return None

    # ── Capture silver_load_timestamp BEFORE groupBy ───────────────────────
    silver_ts_row = silver.agg(
        F.max("silver_load_timestamp").alias("max_ts")
    ).collect()[0]
    silver_ts_str = str(silver_ts_row["max_ts"])

    load_ts_str   = str(datetime.now() + timedelta(hours=load_timestamp_offset))

    # ── Aggregate ──────────────────────────────────────────────────────────
    gold = silver.groupBy(
        "warehouse_id",
        "event_type",
        "business_date"
    ).agg(
        F.count("*")
         .cast(LongType())
         .alias("event_count"),
        F.sum(
            F.when(F.col("quantity_change") > 0, F.col("quantity_change"))
             .otherwise(0)
        ).cast(LongType())
         .alias("total_quantity_in"),
        F.abs(
            F.sum(
                F.when(
                    F.col("quantity_change") < 0,
                    F.col("quantity_change")
                ).otherwise(0)
            )
        ).cast(LongType())
         .alias("total_quantity_out"),
        F.sum("quantity_change")
         .cast(LongType())
         .alias("net_quantity_change"),
        F.sum(F.coalesce("total_cost_impact", F.lit(0.0)))
         .alias("total_cost_impact"),
        F.countDistinct("sku")
         .cast(LongType())
         .alias("unique_skus"),
        F.countDistinct("store_id")
         .cast(LongType())
         .alias("unique_stores")
    )

    # ── Add audit columns ──────────────────────────────────────────────────
    gold = gold \
        .withColumn(
            "silver_load_timestamp",
            F.lit(silver_ts_str).cast(TimestampType())
        ).withColumn(
            "gold_load_timestamp",
            F.lit(load_ts_str).cast(TimestampType())
        ).withColumn(
            "pipeline_version",
            F.lit(PIPELINE_VERSION)
        )

    gold_count = gold.count()
    print(
        f"    Gold rows: {gold_count} "
        f"(one per warehouse per event_type per date)"
    )

    return gold


print("Gold aggregation functions loaded.")

## Write Functions

In [0]:
def write_silver_transactions(df: DataFrame, business_date: date):
    """
    Writes Silver transactions to the target table.
    Enforces exact column order and excludes Raw-only columns.
    """
    if df is None:
        print(f"    No Silver transactions to write for {business_date}.")
        return

    # Exact column order matching the DDL
    # Any column not in this list is excluded
    target_cols = [
        "transaction_id",
        "transaction_date",
        "transaction_timestamp",
        "store_id",
        "store_region",
        "terminal_id",
        "cashier_id",
        "customer_id",
        "loyalty_card_number",
        "transaction_type",
        "payment_method",
        "channel",
        "gross_amount",
        "discount_amount",
        "tax_amount",
        "net_amount",
        "currency_code",
        "item_count",
        "quantity_total",
        "is_derived_net_amount",
        "quality_score",
        "source_file_name",
        "silver_load_timestamp",
        "pipeline_version"
    ]

    # Select only columns that exist in both the list and the DataFrame
    # This handles cases where optional columns may be missing
    available = [c for c in target_cols if c in df.columns]

    # Add any missing columns as null with correct type
    df_write = df.select(available)

    # Add missing columns with null values if not present
    if "is_derived_net_amount" not in df.columns:
        df_write = df_write.withColumn(
            "is_derived_net_amount", F.lit(False).cast(BooleanType())
        )
    if "quality_score" not in df.columns:
        df_write = df_write.withColumn(
            "quality_score", F.lit(1.0).cast(DoubleType())
        )

    # Re-select in exact order after adding missing columns
    df_write = df_write.select(target_cols)

    df_write.write \
            .mode("append") \
            .partitionBy("transaction_date") \
            .saveAsTable(SILVER_TRANSACTIONS)

    count = df_write.count()
    print(f"    ✅ Written {count} records to {SILVER_TRANSACTIONS}")


def write_silver_inventory(df: DataFrame, business_date: date):
    """
    Writes Silver inventory to the target table.
    Enforces exact column order and excludes Raw-only columns.
    """
    if df is None:
        print(f"    No Silver inventory to write for {business_date}.")
        return

    target_cols = [
        "event_id",
        "event_timestamp",
        "event_type",
        "business_date",
        "warehouse_id",
        "store_id",
        "location_code",
        "sku",
        "product_id",
        "category_code",
        "supplier_id",
        "quantity_change",
        "quantity_before",
        "quantity_after",
        "unit_cost",
        "total_cost_impact",
        "reference_document_id",
        "reference_document_type",
        "cdc_operation",
        "cdc_source_system",
        "cdc_sequence_number",
        "is_deduped",
        "quality_flag",
        "silver_load_timestamp",
        "pipeline_version"
    ]

    # Add missing columns with defaults
    if "is_deduped" not in df.columns:
        df = df.withColumn("is_deduped", F.lit(True).cast(BooleanType()))
    if "quality_flag" not in df.columns:
        df = df.withColumn("quality_flag", F.lit("CLEAN"))

    available = [c for c in target_cols if c in df.columns]
    df_write  = df.select(available)

    df_write.write \
            .mode("append") \
            .partitionBy("business_date") \
            .saveAsTable(SILVER_INVENTORY)

    count = df_write.count()
    print(f"    ✅ Written {count} records to {SILVER_INVENTORY}")


def write_gold_store_metrics(df: DataFrame, business_date: date):
    """
    Writes Gold store metrics to the target table.
    Enforces exact column order.
    """
    if df is None:
        print(f"    No Gold store metrics to write for {business_date}.")
        return

    target_cols = [
        "store_id",
        "store_region",
        "business_date",
        "total_transactions",
        "total_gross_amount",
        "total_discount_amount",
        "total_net_amount",
        "total_tax_amount",
        "avg_basket_value",
        "total_items_sold",
        "unique_customers",
        "return_count",
        "return_rate",
        "silver_load_timestamp",
        "gold_load_timestamp",
        "pipeline_version"
    ]

    available = [c for c in target_cols if c in df.columns]
    df_write  = df.select(available)

    df_write.write \
            .mode("append") \
            .partitionBy("business_date") \
            .saveAsTable(GOLD_STORE_METRICS)

    count = df_write.count()
    print(f"    ✅ Written {count} records to {GOLD_STORE_METRICS}")


def write_gold_inventory_summary(df: DataFrame, business_date: date):
    """
    Writes Gold inventory summary to the target table.
    Enforces exact column order.
    """
    if df is None:
        print(f"    No Gold inventory summary to write for {business_date}.")
        return

    target_cols = [
        "warehouse_id",
        "event_type",
        "business_date",
        "event_count",
        "total_quantity_in",
        "total_quantity_out",
        "net_quantity_change",
        "total_cost_impact",
        "unique_skus",
        "unique_stores",
        "silver_load_timestamp",
        "gold_load_timestamp",
        "pipeline_version"
    ]

    available = [c for c in target_cols if c in df.columns]
    df_write  = df.select(available)

    df_write.write \
            .mode("append") \
            .partitionBy("business_date") \
            .saveAsTable(GOLD_INV_SUMMARY)

    count = df_write.count()
    print(f"    ✅ Written {count} records to {GOLD_INV_SUMMARY}")


print("Write functions loaded.")

## Batch 1: Historical Clean Data

Processes ONLY the true historical baseline period.
Dates 2026-05-26 to 2026-06-17 (23 days of actual data).
2026-06-18 has no raw data and is automatically skipped.
Controlled anomaly dates (2026-06-19 to 2026-06-24)
are handled by Cells 7 through 12.

In [0]:
print("Generating historical Silver and Gold...")
print()

# ── True historical window only ────────────────────────────────────────────
# Stops at 2026-06-17 which is the last clean historical date
# 2026-06-19 onwards are controlled anomaly batches
# handled by separate cells with specific injection parameters

historical_dates = [
    get_date(offset)
    for offset in range(-37, -7)    # 2026-05-19 to 2026-06-17
    if get_date(offset) <= date(2026, 6, 17)
]

print(f"Historical dates to process: {len(historical_dates)}")
print(f"From: {historical_dates[0]}")
print(f"To  : {historical_dates[-1]}")
print()

total_dates = len(historical_dates)
dates_done  = 0

for business_date in historical_dates:
    print(f"  [{dates_done + 1}/{total_dates}] {business_date}")

    silver_txn = transform_raw_to_silver_transactions(business_date)
    write_silver_transactions(silver_txn, business_date)

    silver_inv = transform_raw_to_silver_inventory(business_date)
    write_silver_inventory(silver_inv, business_date)

    gold_txn = aggregate_silver_to_gold_transactions(business_date)
    write_gold_store_metrics(gold_txn, business_date)

    gold_inv = aggregate_silver_to_gold_inventory(business_date)
    write_gold_inventory_summary(gold_inv, business_date)

    dates_done += 1
    print()

print(f"Batch 1 complete. {dates_done} historical dates processed.")
print("Expected: All reconciliation checks PASS on these dates.")

## Batch 2: Clean Live Day

Clean Silver and Gold for 2026-06-19.
Agent 1 also passed on this date.
All reconciliation checks should PASS.

In [0]:
business_date = BATCH_DATES["clean"]
print(f"Generating clean live batch: {business_date}")
print()

silver_txn = transform_raw_to_silver_transactions(business_date)
write_silver_transactions(silver_txn, business_date)

silver_inv = transform_raw_to_silver_inventory(business_date)
write_silver_inventory(silver_inv, business_date)

gold_txn = aggregate_silver_to_gold_transactions(business_date)
write_gold_store_metrics(gold_txn, business_date)

gold_inv = aggregate_silver_to_gold_inventory(business_date)
write_gold_inventory_summary(gold_inv, business_date)

print()
print("Batch 2 complete.")
print("Expected: All reconciliation checks PASS.")
print("Agent 1 also PASS on this date.")

## Batch 3: Raw Issue Propagates Correctly

Date: 2026-06-20
Agent 1 flagged VOLUME_ANOMALY CRITICAL on transactions
(only 150 raw rows, 77% below baseline).

Silver correctly reflects the reduced raw volume.
Gold correctly reflects reduced Silver.
Reconciliation checks should PASS because the
Raw to Silver survival rate is still ~92%.

Agent 2 will read Agent 1 context and explain:
"Silver and Gold counts are low but consistent
 with the Raw volume drop already flagged by Agent 1."

In [0]:
business_date = BATCH_DATES["raw_issue"]
print(f"Generating batch 3 (raw issue propagates): {business_date}")
print()
print("Note: Raw has only 150 rows on this date (Agent 1 CRITICAL).")
print("Silver and Gold will reflect this reduced volume correctly.")
print("Reconciliation survival rate should still be ~92%.")
print()

# No extra drop or multiplier — Silver correctly mirrors Raw
silver_txn = transform_raw_to_silver_transactions(business_date)
write_silver_transactions(silver_txn, business_date)

silver_inv = transform_raw_to_silver_inventory(business_date)
write_silver_inventory(silver_inv, business_date)

gold_txn = aggregate_silver_to_gold_transactions(business_date)
write_gold_store_metrics(gold_txn, business_date)

gold_inv = aggregate_silver_to_gold_inventory(business_date)
write_gold_inventory_summary(gold_inv, business_date)

print()
print("Batch 3 complete.")
print("Expected: Reconciliation checks PASS.")
print("         Agent 2 notes Agent 1 already flagged Raw volume issue.")

## Batch 4: Unexpected Silver Drop

Date: 2026-06-21
Agent 1 flagged NULL_SPIKE on net_amount (transactions)
and CROSS_COLUMN on qty_after (inventory).

We inject an ADDITIONAL 30% drop in Silver transactions
beyond what transformation rules would naturally cause.
This simulates a Silver pipeline bug that over-filters records.

Expected Agent 2 detection:
  RAW_TO_SILVER row count check → CRITICAL
  Observed survival: ~64% (0.92 × 0.70)
  Expected survival: ~92%
  Deviation: ~30% below expected → CRITICAL

In [0]:
business_date = BATCH_DATES["silver_drop"]
print(f"Generating batch 4 (unexpected Silver drop): {business_date}")
print()
print("Injecting 30% additional Silver drop beyond normal transformation.")
print("Simulates Silver pipeline over-filtering bug.")
print()

# Transactions: inject 30% extra drop in Silver
silver_txn = transform_raw_to_silver_transactions(
    business_date,
    extra_drop_rate=0.30   # ← inject anomaly
)
write_silver_transactions(silver_txn, business_date)

# Inventory: clean Silver (anomaly only in transactions)
silver_inv = transform_raw_to_silver_inventory(business_date)
write_silver_inventory(silver_inv, business_date)

gold_txn = aggregate_silver_to_gold_transactions(business_date)
write_gold_store_metrics(gold_txn, business_date)

gold_inv = aggregate_silver_to_gold_inventory(business_date)
write_gold_inventory_summary(gold_inv, business_date)

print()
print("Batch 4 complete.")
print("Expected: RAW_TO_SILVER row count → CRITICAL on transactions")
print("         Observed survival: ~64%   Expected: ~92%")

## Batch 5: Gold Aggregate Mismatch

Date: 2026-06-22
Agent 1 flagged CATEGORICAL_DOMAIN on payment_method.

Silver row counts are correct.
But we inject a 15% reduction in Gold financial metrics
by applying an amount_multiplier of 0.85.
This simulates a Gold aggregation calculation error
(wrong exchange rate, incorrect formula, rounding error).

Expected Agent 2 detection:
  SILVER_TO_GOLD aggregate check → CRITICAL
  Silver SUM(net_amount) vs Gold SUM(total_net_amount)
  Deviation: 15% → CRITICAL (threshold 10%)

In [0]:
business_date = BATCH_DATES["gold_mismatch"]
print(f"Generating batch 5 (Gold aggregate mismatch): {business_date}")
print()
print("Silver row counts: correct (no extra drop)")
print("Gold financials: 15% understated (amount_multiplier=0.85)")
print("Simulates Gold aggregation calculation error.")
print()

# Silver: clean (row counts correct)
silver_txn = transform_raw_to_silver_transactions(business_date)
write_silver_transactions(silver_txn, business_date)

silver_inv = transform_raw_to_silver_inventory(business_date)
write_silver_inventory(silver_inv, business_date)

# Gold: inject 15% financial understatement
gold_txn = aggregate_silver_to_gold_transactions(
    business_date,
    amount_multiplier=0.85   # ← inject anomaly
)
write_gold_store_metrics(gold_txn, business_date)

# Inventory Gold: clean
gold_inv = aggregate_silver_to_gold_inventory(business_date)
write_gold_inventory_summary(gold_inv, business_date)

print()
print("Batch 5 complete.")
print("Expected: SILVER_TO_GOLD aggregate → CRITICAL on transactions")
print("         Silver net_amount sum vs Gold total_net_amount sum: 15% gap")

## Batch 6: Cascading Raw Issue

Date: 2026-06-23
Agent 1 flagged CROSS_COLUMN_CONSISTENCY on transactions
(11.67% of records violate gross - discount + tax = net_amount)
and VOLUME_ANOMALY WARNING (767 rows, 20.56% above baseline).

Silver correctly applies its rules and DERIVES net_amount
for the 11.67% of inconsistent records using the formula.
This means Silver drops fewer records than usual
(none are dropped for inconsistency, they are corrected).

But the derived net_amount values may differ from raw net_amount
causing a Silver to Gold aggregate mismatch that is SMALLER
than the Gold mismatch batch but still detectable.

This is a cascading scenario where a Raw quality issue
causes a downstream reconciliation drift.

In [0]:
business_date = BATCH_DATES["cascading"]
print(f"Generating batch 6 (cascading raw issue): {business_date}")
print()
print("Raw has cross-column inconsistency on net_amount (11.67%).")
print("Silver derives net_amount for inconsistent records.")
print("Gold aggregates derived values.")
print("Small aggregate drift expected between Raw and Gold.")
print()

# Silver: clean transformation (derives net_amount automatically)
silver_txn = transform_raw_to_silver_transactions(business_date)
write_silver_transactions(silver_txn, business_date)

silver_inv = transform_raw_to_silver_inventory(
    business_date,
    extra_drop_rate=0.05   # Extra 5% drop for inventory (over-filter)
)
write_silver_inventory(silver_inv, business_date)

gold_txn = aggregate_silver_to_gold_transactions(business_date)
write_gold_store_metrics(gold_txn, business_date)

gold_inv = aggregate_silver_to_gold_inventory(business_date)
write_gold_inventory_summary(gold_inv, business_date)

print()
print("Batch 6 complete.")
print("Expected: Small aggregate drift on transactions (cascading from Raw)")
print("         Inventory Silver drop slightly below expected survival rate")
print("         Agent 2 reads Agent 1 context to explain the cascade")

## Batch 7: Layer Freshness Issue + Combined Problems

Date: 2026-06-24
Agent 1 flagged multiple CRITICAL issues on both tables.

We inject:
  1. Silver transactions written with 3-hour delay
     (simulates Silver pipeline running late)
  2. Gold inventory NOT written at all
     (simulates Gold pipeline failure)
  3. Silver inventory has additional 20% drop
     (compounding the Raw event_id null issue)

Expected Agent 2 detection:
  Freshness check on Silver transactions → WARNING (3hr > 2hr threshold)
  Gold inventory missing → CRITICAL (no rows for this date)
  Silver inventory survival → CRITICAL (compounding Raw issue)

In [0]:
business_date = BATCH_DATES["freshness_issue"]
print(f"Generating batch 7 (freshness + combined): {business_date}")
print()
print("Injecting:")
print("  Silver transactions: 3-hour load delay")
print("  Silver inventory: additional 20% drop")
print("  Gold inventory: NOT written (pipeline failure simulation)")
print()

# Silver transactions: correct data but written 3 hours late
silver_txn = transform_raw_to_silver_transactions(
    business_date,
    load_timestamp_offset=-3   # Negative = timestamp 3 hours in the past
                               # Simulates late arrival
)
write_silver_transactions(silver_txn, business_date)

# Silver inventory: additional 20% drop (compounding raw null issue)
silver_inv = transform_raw_to_silver_inventory(
    business_date,
    extra_drop_rate=0.20
)
write_silver_inventory(silver_inv, business_date)

# Gold transactions: clean (only Silver latency is the issue here)
gold_txn = aggregate_silver_to_gold_transactions(business_date)
write_gold_store_metrics(gold_txn, business_date)

# Gold inventory: NOT written (simulates Gold pipeline failure)
print("  Skipping Gold inventory write — simulating pipeline failure.")

print()
print("Batch 7 complete.")
print("Expected:")
print("  Freshness check transactions Silver → WARNING (3hr delay)")
print("  Silver inventory survival → CRITICAL (compounding)")
print("  Gold inventory missing → CRITICAL (no rows found)")
print("  AI invocation → TRIGGERED (multiple failures, cascading context)")

## Verification: Confirm All Layers Generated Correctly

In [0]:
print("=" * 65)
print("SILVER AND GOLD GENERATION VERIFICATION")
print("=" * 65)

# Row counts across all layers by date
print("\n1. Row counts across layers:\n")
spark.sql(f"""
    SELECT
        r.transaction_date              AS business_date,
        COUNT(r.transaction_id)         AS raw_count,
        COUNT(s.transaction_id)         AS silver_count,
        COUNT(g.store_id)               AS gold_rows,
        ROUND(
            COUNT(s.transaction_id) /
            NULLIF(COUNT(r.transaction_id), 0) * 100, 2
        )                               AS raw_to_silver_pct
    FROM learning_dev.s819534.uc1_raw_store_transactions r
    LEFT JOIN learning_dev.s819534.uc2_silver_store_transactions s
        ON  r.transaction_id    = s.transaction_id
        AND r.transaction_date  = s.transaction_date
    LEFT JOIN (
        SELECT business_date, SUM(total_transactions) AS store_id
        FROM learning_dev.s819534.uc2_gold_daily_store_metrics
        GROUP BY business_date
    ) g ON r.transaction_date = g.business_date
    GROUP BY r.transaction_date
    ORDER BY r.transaction_date
""").show(50, truncate=False)

# Aggregate comparison
print("\n2. Financial aggregate comparison (transactions):\n")
spark.sql(f"""
    SELECT
        r.business_date,
        ROUND(r.raw_net,    2) AS raw_net_amount,
        ROUND(s.silver_net, 2) AS silver_net_amount,
        ROUND(g.gold_net,   2) AS gold_net_amount,
        ROUND(
            ABS(s.silver_net - g.gold_net) /
            NULLIF(s.silver_net, 0) * 100, 4
        )                      AS silver_gold_deviation_pct
    FROM (
        SELECT transaction_date AS business_date,
               SUM(net_amount) AS raw_net
        FROM learning_dev.s819534.uc1_raw_store_transactions
        GROUP BY transaction_date
    ) r
    LEFT JOIN (
        SELECT transaction_date AS business_date,
               SUM(net_amount) AS silver_net
        FROM learning_dev.s819534.uc2_silver_store_transactions
        GROUP BY transaction_date
    ) s ON r.business_date = s.business_date
    LEFT JOIN (
        SELECT business_date,
               SUM(total_net_amount) AS gold_net
        FROM learning_dev.s819534.uc2_gold_daily_store_metrics
        GROUP BY business_date
    ) g ON r.business_date = g.business_date
    ORDER BY r.business_date
""").show(50, truncate=False)

# Inventory counts
print("\n3. Inventory movement counts across layers:\n")
spark.sql(f"""
    SELECT
        r.business_date,
        r.raw_count,
        COALESCE(s.silver_count, 0) AS silver_count,
        COALESCE(g.gold_rows, 0)    AS gold_warehouse_event_rows,
        ROUND(
            COALESCE(s.silver_count, 0) /
            NULLIF(r.raw_count, 0) * 100, 2
        )                           AS survival_pct
    FROM (
        SELECT business_date, COUNT(*) AS raw_count
        FROM learning_dev.s819534.uc1_raw_inventory_movements
        GROUP BY business_date
    ) r
    LEFT JOIN (
        SELECT business_date, COUNT(*) AS silver_count
        FROM learning_dev.s819534.uc2_silver_inventory_movements
        GROUP BY business_date
    ) s ON r.business_date = s.business_date
    LEFT JOIN (
        SELECT business_date, COUNT(*) AS gold_rows
        FROM learning_dev.s819534.uc2_gold_daily_inventory_summary
        GROUP BY business_date
    ) g ON r.business_date = g.business_date
    ORDER BY r.business_date
""").show(50, truncate=False)

# Summary
print("\n4. Table row counts summary:\n")
for tbl in [
    "uc1_raw_store_transactions",
    "uc2_silver_store_transactions",
    "uc2_gold_daily_store_metrics",
    "uc1_raw_inventory_movements",
    "uc2_silver_inventory_movements",
    "uc2_gold_daily_inventory_summary"
]:
    count = spark.table(f"learning_dev.s819534.{tbl}").count()
    print(f"  {tbl:<45} {count:>8} rows")

## Final Summary

In [0]:
print("\n" + "=" * 65)
print("SILVER AND GOLD GENERATION COMPLETE")
print("=" * 65)
print()
print("Batches generated:")
print()
print(f"  Historical (30 days) : Clean Silver and Gold")
print(f"  {BATCH_DATES['clean']}           : Clean live day")
print(f"  {BATCH_DATES['raw_issue']}       : Raw issue propagates correctly")
print(f"                           (Agent 2 PASS, Agent 1 CRITICAL)")
print(f"  {BATCH_DATES['silver_drop']}     : Silver unexpected drop")
print(f"                           (Raw→Silver CRITICAL)")
print(f"  {BATCH_DATES['gold_mismatch']}   : Gold aggregate mismatch")
print(f"                           (Silver→Gold aggregate CRITICAL)")
print(f"  {BATCH_DATES['cascading']}       : Cascading raw quality issue")
print(f"                           (Small aggregate drift)")
print(f"  {BATCH_DATES['freshness_issue']} : Freshness + combined issues")
print(f"                           (Latency WARNING, missing Gold CRITICAL)")
print()
print("Next step:")
print("  Build Agent 2 reconciliation notebook")
print("  02_reconciliation_agent")
print("=" * 65)